In [1]:
import os
import json
import re
import hashlib
from pathlib import Path

import pandas as pd



In [2]:
DATASET_ROOT_CANDIDATES = [
    Path("../input/AI4Code"),
    Path("/kaggle/input/AI4Code"),
    Path("/kaggle/input/ai4code"),  # just in case
    Path("../input/ai4code"),
]

dataset_root = None
for p in DATASET_ROOT_CANDIDATES:
    if (p / "sample_submission.csv").exists() and (p / "test").exists():
        dataset_root = p
        break

if dataset_root is None:
    for p in [Path("/kaggle/input"), Path("../input")]:
        if p.exists():
            for child in p.iterdir():
                if (
                    child.is_dir()
                    and (child / "sample_submission.csv").exists()
                    and (child / "test").exists()
                ):
                    dataset_root = child
                    break
        if dataset_root is not None:
            break

if dataset_root is None:
    raise FileNotFoundError(
        "Could not locate dataset root containing sample_submission.csv and test/ directory."
    )

print("Using dataset_root:", dataset_root)



Using dataset_root: ../input/AI4Code


In [3]:
METRIC_SPLIT_MODE = "max"
ENSEMBLE_MODE = "weighted_average"



In [4]:
IN_FEATURIZE = False



In [5]:
sample_path = dataset_root / "sample_submission.csv"
test_dir = dataset_root / "test"

sample_df = pd.read_csv(sample_path)
print("sample_submission shape:", sample_df.shape)
print(sample_df.head())

assert list(sample_df.columns) == [
    "id",
    "cell_order",
], "Unexpected submission format in sample_submission.csv"
assert test_dir.exists(), f"Missing test directory: {test_dir}"



sample_submission shape: (20000, 2)
               id                                         cell_order
0  00015c83e2717b  c417225b 51e3cd89 2600b4eb 75b65993 cf195f8b 2...
1  0001bdd4021779  3fdc37be 073782ca 8ea7263c 80543cd8 38310c80 0...
2  000757b90aaca0  8f84d7a9 eb6ca769 bc595bc2 93cceeef 3cb3d383 6...
3  000a2f5243e1ca  1d968d84 5774aca9 2ddb979d dc18005d 3c7d2db1 2...
4  000c1e0e45bb25  3255fff2 d6c42868 dd65465b 07c5250a 31aafb08 8...


In [6]:
_word_re = re.compile(r"[a-z0-9]+", re.IGNORECASE)

_code_kw_re = re.compile(
    r"\b(import|from|pip|conda|install|load|read_csv|data|dataset|preprocess|clean|feature|"
    r"train|fit|model|predict|inference|evaluate|metric|score|loss|accuracy|auc|roc|"
    r"plot|plt\.|seaborn|visuali[sz]e|figure|table|submission)\b",
    re.IGNORECASE,
)
_title_like_re = re.compile(r"^\s{0,3}#{1,3}\s+\S+", re.IGNORECASE)

_CODE_PHASE_PATTERNS = [
    (
        "setup",
        re.compile(
            r"\b(import|from|pip|conda|install|setup|requirements|environment)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "data",
        re.compile(
            r"\b(data|dataset|read_csv|read_parquet|load|preprocess|clean|feature|tokeni[sz]e|split|train_test_split)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "eda",
        re.compile(
            r"\b(eda|explor|distribution|summary|describe\(|head\(|value_counts|missing|null|nan)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "train",
        re.compile(
            r"\b(train|fit|training|cross[-\s]?val|validation|fold|kfold|stratif|baseline|pipeline)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "eval",
        re.compile(
            r"\b(evaluate|metric|score|loss|accuracy|auc|roc|confusion|rmse|mae|logloss)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "viz",
        re.compile(
            r"\b(plot|plt\.|seaborn|visuali[sz]e|figure|table|chart|graph|heatmap|pairplot)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "submit",
        re.compile(
            r"\b(submission|infer|inference|predict|test set|predicting)\b",
            re.IGNORECASE,
        ),
    ),
]

_LATE_SECTION_RE = re.compile(
    r"\b(conclusion|summary|discussion|future work|references|bibliography|appendix|acknowledg)\b",
    re.IGNORECASE,
)
_EARLY_SECTION_RE = re.compile(
    r"\b(introduction|overview|background|context|objective|aim|problem statement)\b",
    re.IGNORECASE,
)

_STRICT_REF_RE = re.compile(
    r"^\s{0,3}#{0,3}\s*(references|bibliography)\s*$", re.IGNORECASE
)
_STRICT_APPENDIX_RE = re.compile(r"^\s{0,3}#{0,3}\s*appendix\b", re.IGNORECASE)
_STRICT_ACK_RE = re.compile(r"^\s{0,3}#{0,3}\s*acknowledg", re.IGNORECASE)

# Change (score-toward-target): explicit section/step ordering cues are strong for markdown ordering.
# We only add a tiny adjustment to the existing md score (no architecture change).
_STEP_NUM_RE = re.compile(
    r"\b(step|part|section|chapter)\s*([0-9]{1,2})\b", re.IGNORECASE
)
_ROMAN_RE = re.compile(r"\b(part|section|chapter)\s*([ivxlcdm]{1,6})\b", re.IGNORECASE)


def _stable_u01_from_str(s: str) -> float:
    """Deterministic tie-break/jitter in [0,1) from cell_id."""
    h = hashlib.md5((s or "").encode("utf-8")).digest()
    v = int.from_bytes(h[:4], "little", signed=False)
    return (v % 10_000_000) / 10_000_000.0


def _md_pos_score(md_text: str) -> float:
    """
    Keep same markdown position score idea in [0,1] with lightweight cues.
    """
    t = (md_text or "").strip()
    t_l = t.lower()

    if t_l.startswith("# "):
        base = 0.05
    elif t_l.startswith("## "):
        base = 0.12
    elif t_l.startswith("### "):
        base = 0.18
    else:
        base = 0.30

    early_kw = (
        "overview",
        "introduction",
        "background",
        "context",
        "objective",
        "aim",
        "problem",
        "description",
        "dataset",
        "data",
        "setup",
        "imports",
    )
    mid_kw = (
        "method",
        "approach",
        "model",
        "training",
        "experiment",
        "analysis",
        "results",
        "evaluation",
    )
    late_kw = (
        "conclusion",
        "summary",
        "discussion",
        "future",
        "reference",
        "references",
        "appendix",
        "acknowledg",
    )

    if any(k in t_l for k in early_kw):
        base -= 0.12
    elif any(k in t_l for k in late_kw):
        base += 0.45
    elif any(k in t_l for k in mid_kw):
        base += 0.12

    output_kw = (
        "figure",
        "fig.",
        "plot",
        "chart",
        "graph",
        "table",
        "output",
        "result",
        "results",
        "accuracy",
        "roc",
        "auc",
        "loss",
        "confusion",
        "metric",
    )
    if any(k in t_l for k in output_kw):
        base += 0.08

    step_early = ("first", "step 1", "step one", "to begin", "we start", "initially")
    step_mid = ("next", "then", "after", "in this section", "now", "let us", "we will")
    step_late = ("finally", "in conclusion", "to summarize", "overall")

    if any(k in t_l for k in step_early):
        base -= 0.04
    if any(k in t_l for k in step_mid):
        base += 0.03
    if any(k in t_l for k in step_late):
        base += 0.06

    n_chars = len(t)
    if n_chars <= 50:
        base -= 0.05
    elif n_chars >= 400:
        base += 0.05

    if (
        t_l.startswith("#") or t_l.startswith("##") or t_l.startswith("###")
    ) and n_chars <= 30:
        base += 0.03

    if t_l in ("references", "# references", "## references"):
        base = 0.98

    if base < 0.0:
        base = 0.0
    if base > 1.0:
        base = 1.0
    return float(base)


def _md_anchor_adjustment(md_text: str) -> float:
    """
    Small adjustment to md score based on explicit code-related keywords.
    """
    t = (md_text or "").strip()
    if not t:
        return 0.0

    hits = len(_code_kw_re.findall(t))
    if hits <= 0:
        return 0.0

    return min(0.10, 0.04 + 0.02 * (hits - 1))


def _code_phase_anchors(code_ids, source_dict):
    n_code = len(code_ids)
    if n_code <= 0:
        return {}

    hits = {name: [] for name, _ in _CODE_PHASE_PATTERNS}
    for i, cid in enumerate(code_ids):
        t = source_dict.get(cid, "") or ""
        for name, pat in _CODE_PHASE_PATTERNS:
            if pat.search(t):
                hits[name].append(i)

    anchors = {}
    for name, idxs in hits.items():
        if not idxs:
            continue
        idxs_sorted = sorted(idxs)
        anchors[name] = idxs_sorted[len(idxs_sorted) // 2]  # median index
    return anchors


def _md_phase_target(md_text: str, anchors: dict):
    t = (md_text or "").strip()
    if not t or not anchors:
        return None

    best = None
    best_hits = 0
    for name, pat in _CODE_PHASE_PATTERNS:
        if name not in anchors:
            continue
        c = len(pat.findall(t))
        if c > best_hits:
            best_hits = c
            best = name
    if best is None or best_hits <= 0:
        return None
    return anchors[best]


def _token_set(text: str):
    return set(_word_re.findall((text or "").lower()))


def _md_near_code_adjustment(
    md_text: str, code_ids, source_dict, anchors=None
) -> float:
    """
    Returns a small delta in [-0.06, +0.06] to apply to base score based on which code index
    has the highest token overlap with the markdown. Deterministic and lightweight.

    Probes deterministic code positions and also probes around the strongest phase anchor
    (if available) to reduce local inversions.
    """
    n_code = len(code_ids)
    if n_code <= 0:
        return 0.0
    md_toks = _token_set(md_text)
    if not md_toks:
        return 0.0

    probes = set([0, n_code - 1])
    probes.update(int(i * (n_code - 1) / 5) for i in range(1, 5))
    probes.update(int(i * (n_code - 1) / 4) for i in range(1, 4))

    if anchors:
        best_name = None
        best_hits = 0
        t = md_text or ""
        for name, pat in _CODE_PHASE_PATTERNS:
            if name in anchors:
                c = len(pat.findall(t))
                if c > best_hits:
                    best_hits = c
                    best_name = name
        if best_name is not None and best_hits > 0:
            a = int(anchors[best_name])
            for j in (a - 2, a - 1, a, a + 1, a + 2):
                if 0 <= j < n_code:
                    probes.add(j)

    probes = sorted(probes)

    best_i = None
    best_overlap = 0

    for i in probes:
        cid = code_ids[i]
        code_toks = _token_set(source_dict.get(cid, ""))
        if not code_toks:
            continue
        overlap = len(md_toks & code_toks)
        if overlap > best_overlap:
            best_overlap = overlap
            best_i = i

    if best_i is None or best_overlap <= 0:
        return 0.0

    tgt = (best_i + 0.5) / max(1, n_code)
    delta = (tgt - 0.5) * 0.12  # max magnitude ~0.06
    if delta < -0.06:
        delta = -0.06
    if delta > 0.06:
        delta = 0.06
    return float(delta)


def _code_third_token_sets(code_ids, source_dict):
    """
    Derive a very lightweight notebook-local structural signal from code that is already in
    correct order. We compute token sets for early/mid/late thirds of the code and use markdown
    overlap with these to gently pull markdown earlier/later.
    """
    n = len(code_ids)
    if n <= 0:
        return None

    def _span_tokens(a, b):
        toks = set()
        for i in range(a, b):
            toks |= _token_set(source_dict.get(code_ids[i], ""))
            if len(toks) > 5000:  # hard cap for speed/memory determinism
                break
        return toks

    s1 = n // 3
    s2 = (2 * n) // 3
    if s1 <= 0:
        s1 = 1 if n >= 1 else 0
    if s2 <= s1:
        s2 = min(n, s1 + 1)

    early = _span_tokens(0, s1)
    mid = _span_tokens(s1, s2)
    late = _span_tokens(s2, n)
    return early, mid, late


def _md_code_context_adjustment(md_text: str, thirds) -> float:
    """
    Small delta in [-0.05, +0.05] based on whether markdown overlaps more with early vs late
    code thirds.
    """
    if not thirds:
        return 0.0
    early, mid, late = thirds
    md_toks = _token_set(md_text)
    if not md_toks:
        return 0.0

    e = len(md_toks & early) if early else 0
    m = len(md_toks & mid) if mid else 0
    l = len(md_toks & late) if late else 0
    tot = e + m + l
    if tot <= 0:
        return 0.0

    bias = (l - e) / max(1, tot)
    delta = bias * 0.05
    if delta < -0.05:
        delta = -0.05
    if delta > 0.05:
        delta = 0.05
    return float(delta)


def _md_local_window_adjustment(md_text: str, code_ids, source_dict) -> float:
    """
    Change (score-toward-target): tiny refinement that looks at a small window around the
    best-overlap code index (computed with full scan only in a tiny window) and nudges the
    markdown score slightly toward that window's center. This reduces local inversions while
    preserving the same score->bucket approach.

    Change vs previous: slightly widen the window (still fixed small) to better resolve local
    maxima without an expensive full scan.
    """
    n_code = len(code_ids)
    if n_code <= 0:
        return 0.0
    md_toks = _token_set(md_text)
    if not md_toks:
        return 0.0

    probes = [0, n_code - 1]
    probes += [int(i * (n_code - 1) / 5) for i in range(1, 5)]
    probes = sorted(set(probes))

    best_i = None
    best_overlap = 0
    for i in probes:
        code_toks = _token_set(source_dict.get(code_ids[i], ""))
        if not code_toks:
            continue
        overlap = len(md_toks & code_toks)
        if overlap > best_overlap:
            best_overlap = overlap
            best_i = i

    if best_i is None or best_overlap <= 1:
        return 0.0

    w = 4
    a = max(0, best_i - w)
    b = min(n_code, best_i + w + 1)
    local_best_i = best_i
    local_best_overlap = best_overlap
    for i in range(a, b):
        code_toks = _token_set(source_dict.get(code_ids[i], ""))
        if not code_toks:
            continue
        overlap = len(md_toks & code_toks)
        if overlap > local_best_overlap:
            local_best_overlap = overlap
            local_best_i = i

    tgt = (local_best_i + 0.5) / max(1, n_code)
    delta = (tgt - 0.5) * 0.04  # max magnitude ~0.02
    if delta < -0.02:
        delta = -0.02
    if delta > 0.02:
        delta = 0.02
    return float(delta)


def _is_strict_reference_like(md_text: str) -> bool:
    t = (md_text or "").strip()
    if not t:
        return False
    tl = t.lower()
    if _STRICT_REF_RE.match(t):
        return True
    if _STRICT_APPENDIX_RE.match(t):
        return True
    if _STRICT_ACK_RE.match(t):
        return True
    if tl in (
        "references",
        "# references",
        "## references",
        "bibliography",
        "# bibliography",
        "## bibliography",
    ):
        return True
    return False


def _apply_md_order_prior(md_base: dict, md_ids: list, strength: float = 0.06) -> dict:
    """
    Change (score-toward-target): add a *weak* prior based on the markdown order as it appears
    in the JSON (shuffled but still carries some adjacency coherence). This does NOT replace
    text scoring; it only slightly pulls scores toward the md's relative index to reduce
    scatter-driven inversions.
    """
    n = len(md_ids)
    if n <= 1:
        return md_base
    out = dict(md_base)
    for i, cid in enumerate(md_ids):
        if cid not in out:
            continue
        prior = (i + 0.5) / n  # (0,1)
        out[cid] = (1.0 - strength) * out[cid] + strength * prior
    return out


def _pairwise_coherence_key(md_base: dict, md_pos: dict, cid: str, eps: float = 0.02):
    """
    Change (score-toward-target): deterministic ordering key that keeps markdowns with very
    similar scores closer to their original md order, reducing local swaps when scores tie/cluster.
    """
    b = md_base.get(cid, 0.5)
    band = int(b / eps)  # coarse band id
    return (band, b, md_pos.get(cid, 0), cid)


def _md_continuous_bucket_key(md_base: dict, md_pos: dict, cid: str, n_code: int):
    """
    Change (score-toward-target): stabilize ordering near bucket boundaries by sorting markdowns
    using their implied continuous x=score*n_code before the existing tie-breaks.
    """
    b = md_base.get(cid, 0.5)
    x = b * max(1, n_code)
    return (x, b, md_pos.get(cid, 0), cid)


def _code_prefix_suffix_token_sets(code_ids, source_dict):
    """
    Change (score-toward-target): tiny extra notebook-local signal. Compute token sets for
    a small prefix and suffix of code cells (code is already ordered correctly).
    """
    n = len(code_ids)
    if n <= 0:
        return None

    k = min(12, n)  # fixed small cap for speed/determinism

    prefix = set()
    for i in range(0, k):
        prefix |= _token_set(source_dict.get(code_ids[i], ""))
        if len(prefix) > 6000:
            break

    suffix = set()
    for i in range(max(0, n - k), n):
        suffix |= _token_set(source_dict.get(code_ids[i], ""))
        if len(suffix) > 6000:
            break

    return prefix, suffix


def _md_prefix_suffix_adjustment(md_text: str, pref_suf) -> float:
    """
    Change (score-toward-target): small delta in [-0.03, +0.03] using prefix vs suffix overlap.
    """
    if not pref_suf:
        return 0.0
    prefix, suffix = pref_suf
    md_toks = _token_set(md_text)
    if not md_toks:
        return 0.0

    p = len(md_toks & prefix) if prefix else 0
    s = len(md_toks & suffix) if suffix else 0
    tot = p + s
    if tot <= 0:
        return 0.0

    bias = (s - p) / max(1, tot)  # in [-1,1]
    delta = bias * 0.03
    if delta < -0.03:
        delta = -0.03
    if delta > 0.03:
        delta = 0.03
    return float(delta)


def _code_kw_centroid(code_ids, source_dict):
    """
    Change (score-toward-target): compute a cheap "centroid" of where code-keyword-heavy cells occur.
    Markdown with code keywords tends to align to where code intensity is higher; this helps reduce
    systematic early/late inversions without changing the core score->bucket approach.
    Returns normalized centroid in [0,1], or None if no signal.
    """
    n = len(code_ids)
    if n <= 0:
        return None

    weights = []
    idxs = []
    for i, cid in enumerate(code_ids):
        t = source_dict.get(cid, "") or ""
        w = len(_code_kw_re.findall(t))
        if w > 0:
            weights.append(min(6, w))
            idxs.append(i)

    if not weights:
        return None

    num = 0.0
    den = 0.0
    for w, i in zip(weights, idxs):
        num += w * (i + 0.5)
        den += w
    c = num / max(1e-9, den)  # in [0.5, n-0.5]
    return float(c / max(1, n))  # ~ (0,1)


def _md_kw_centroid_adjustment(md_text: str, kw_centroid) -> float:
    """
    Change (score-toward-target): small delta in [-0.03,+0.03] pulling markdown with code keywords
    toward the notebook's code-keyword centroid.
    """
    if kw_centroid is None:
        return 0.0
    t = (md_text or "").strip()
    if not t:
        return 0.0
    hits = len(_code_kw_re.findall(t))
    if hits <= 0:
        return 0.0

    base_pull = (kw_centroid - 0.5) * 0.06  # max +/-0.03
    if base_pull < -0.03:
        base_pull = -0.03
    if base_pull > 0.03:
        base_pull = 0.03
    return float(base_pull)


def _roman_to_int(r: str) -> int:
    r = (r or "").lower()
    vals = {"i": 1, "v": 5, "x": 10, "l": 50, "c": 100, "d": 500, "m": 1000}
    total = 0
    prev = 0
    for ch in reversed(r):
        v = vals.get(ch, 0)
        if v < prev:
            total -= v
        else:
            total += v
            prev = v
    return total


def _md_step_order_adjustment(md_text: str) -> float:
    """
    Change (score-toward-target): tiny delta in [-0.04, +0.06] using explicit 'Step N' / 'Part II'
    cues. This helps place procedural markdown along the notebook timeline without changing the
    core score->bucket logic.
    """
    t = (md_text or "").strip()
    if not t:
        return 0.0

    m = _STEP_NUM_RE.search(t)
    if m:
        try:
            n = int(m.group(2))
        except Exception:
            n = None
        if n is not None and 1 <= n <= 12:
            pos = (n - 1) / 11.0  # map 1..12 -> 0..1
            delta = (pos - 0.35) * 0.08  # keep small
            if delta < -0.04:
                delta = -0.04
            if delta > 0.06:
                delta = 0.06
            return float(delta)

    m = _ROMAN_RE.search(t)
    if m:
        rn = _roman_to_int(m.group(2))
        if 1 <= rn <= 12:
            pos = (rn - 1) / 11.0
            delta = (pos - 0.35) * 0.08
            if delta < -0.04:
                delta = -0.04
            if delta > 0.06:
                delta = 0.06
            return float(delta)

    return 0.0


def _md_soft_late_bias(md_text: str) -> float:
    """
    Change (score-toward-target): soften late bias to avoid misplacing mid-notebook 'discussion/results'
    markdown. We keep strict reference/appendix handling elsewhere; this is only a small score nudge.
    """
    t = (md_text or "").strip()
    if not t:
        return 0.0
    tl = t.lower()

    # High-precision late: very likely end-of-notebook.
    strong = (
        bool(
            re.search(
                r"\b(in conclusion|to conclude|to summarize|summary of|final remarks)\b",
                tl,
            )
        )
        or bool(re.search(r"\b(acknowledg(e)?ments?)\b", tl))
        or bool(re.search(r"\b(bibliograph(y|ies))\b", tl))
    )
    if strong:
        return 0.035

    # If it contains both "results" and "discussion"/"conclusion", do NOT push hard late.
    # This reduces false late pushes for mid-notebook analysis.
    mixed_mid = ("results" in tl) and bool(re.search(r"\b(discussion|analysis)\b", tl))
    if mixed_mid:
        return -0.01

    return 0.0


def load_notebook_cells(nb_path: Path):
    """
    Preserve code order exactly; position markdown via score->bucket mapping.
    """
    with nb_path.open("r", encoding="utf-8") as f:
        nb = json.load(f)

    cell_type = nb["cell_type"]  # dict: cell_id -> "code"/"markdown"
    source = nb.get("source", {})  # dict: cell_id -> str

    cell_ids = list(cell_type.keys())
    code_ids = [cid for cid in cell_ids if cell_type[cid] == "code"]
    md_ids = [cid for cid in cell_ids if cell_type[cid] == "markdown"]

    md_pos = {cid: i for i, cid in enumerate(md_ids)}
    if len(code_ids) == 0:
        md_sorted = sorted(
            md_ids,
            key=lambda cid: (_md_pos_score(source.get(cid, "")), md_pos[cid], cid),
        )
        return md_sorted

    n_code = len(code_ids)
    buckets = [[] for _ in range(n_code + 1)]

    anchors = _code_phase_anchors(code_ids, source)
    code_thirds = _code_third_token_sets(code_ids, source)
    pref_suf = _code_prefix_suffix_token_sets(code_ids, source)

    kw_centroid = _code_kw_centroid(code_ids, source)

    title_candidate = None
    if md_ids:

        def _title_score(cid: str) -> float:
            t = (source.get(cid, "") or "").strip()
            t_l = t.lower()
            is_heading = 1.0 if _title_like_re.match(t) else 0.0
            has_intro = 1.0 if _EARLY_SECTION_RE.search(t_l) else 0.0
            length = len(t)
            short_bonus = 1.0 if length <= 180 else 0.0
            codeish = 1.0 if len(_code_kw_re.findall(t)) > 0 else 0.0
            return (
                -is_heading,
                -has_intro,
                codeish,
                -short_bonus,
                length,
                md_pos[cid],
                cid,
            )

        cand = min(md_ids, key=_title_score)
        cand_text = (source.get(cand, "") or "").strip()
        cand_l = cand_text.lower()
        looks_like_title = bool(
            _title_like_re.match(cand_text)
        )  # stricter: heading required
        if (
            looks_like_title
            and (_EARLY_SECTION_RE.search(cand_l) or cand_text.startswith("#"))
            and not _LATE_SECTION_RE.search(cand_l)
        ):
            title_candidate = cand

    force_late_md = set()
    for cid in md_ids:
        if _is_strict_reference_like(source.get(cid, "") or ""):
            force_late_md.add(cid)

    md_base = {}
    for cid in md_ids:
        md_text = source.get(cid, "") or ""
        base = _md_pos_score(md_text)
        base = base + _md_anchor_adjustment(md_text)

        # Change (score-toward-target): add tiny step/part ordering adjustment to reduce inversions.
        base = base + _md_step_order_adjustment(md_text)

        tgt = _md_phase_target(md_text, anchors)
        if tgt is not None:
            tgt_norm = (tgt + 0.5) / max(1, n_code)  # approx in (0,1)
            base = 0.75 * base + 0.25 * tgt_norm

        base = base + _md_near_code_adjustment(
            md_text, code_ids, source, anchors=anchors
        )
        base = base + _md_code_context_adjustment(md_text, code_thirds)
        base = base + _md_local_window_adjustment(md_text, code_ids, source)
        base = base + _md_prefix_suffix_adjustment(md_text, pref_suf)
        base = base + _md_kw_centroid_adjustment(md_text, kw_centroid)

        # Change (score-toward-target): soften/precision late bias (strict refs still handled separately).
        base = base + _md_soft_late_bias(md_text)

        if base < 0.0:
            base = 0.0
        if base > 1.0:
            base = 1.0

        md_base[cid] = float(base)

    md_base = _apply_md_order_prior(md_base, md_ids, strength=0.06)

    for cid in md_ids:
        if title_candidate is not None and cid == title_candidate:
            buckets[0].append(cid)
            continue

        if cid in force_late_md:
            buckets[n_code].append(cid)
            continue

        base = md_base[cid]

        x = base * n_code
        lo = int(x)  # 0..n_code
        frac = x - lo

        if lo < 0:
            lo = 0
            frac = 0.0
        if lo > n_code:
            lo = n_code
            frac = 0.0

        jitter = _stable_u01_from_str(cid)  # [0,1)
        thresh = 0.50 + (jitter - 0.5) * 0.10  # in [0.45, 0.55]

        if abs(frac - thresh) < 0.04:
            thresh = thresh + 0.02  # slightly harder to push_next

        push_next = frac > thresh
        ins = lo + 1 if (push_next and lo < n_code) else lo
        buckets[ins].append(cid)

    for i in range(n_code + 1):
        buckets[i].sort(
            key=lambda cid: (
                _md_continuous_bucket_key(md_base, md_pos, cid, n_code),
                _pairwise_coherence_key(md_base, md_pos, cid),
            )
        )

    ordered = []
    ordered.extend(buckets[0])
    for i, code_cid in enumerate(code_ids, start=1):
        ordered.append(code_cid)
        ordered.extend(buckets[i])

    ref_like = []
    kept = []
    for cid in ordered:
        if cell_type.get(cid) == "markdown":
            if _is_strict_reference_like(source.get(cid, "") or ""):
                ref_like.append(cid)
                continue
        kept.append(cid)

    if ref_like:
        last_code_idx = -1
        for i in range(len(kept) - 1, -1, -1):
            if cell_type.get(kept[i]) == "code":
                last_code_idx = i
                break
        if last_code_idx == -1:
            ordered = kept + ref_like
        else:
            ordered = kept[: last_code_idx + 1] + ref_like + kept[last_code_idx + 1 :]
    else:
        ordered = kept

    return ordered




In [7]:
pred_orders = {}

missing = 0
for nb_id in sample_df["id"].tolist():
    nb_path = test_dir / f"{nb_id}.json"
    if not nb_path.exists():
        missing += 1
        pred_orders[nb_id] = ""
        continue
    pred_orders[nb_id] = " ".join(load_notebook_cells(nb_path))

print("Missing test notebooks:", missing)
if missing > 0:
    print(
        "WARNING: some test json files were missing; submission will contain empty orders for them."
    )



Missing test notebooks: 0


In [8]:
submission_df = sample_df.copy()
submission_df["cell_order"] = submission_df["id"].map(pred_orders)
submission_df["cell_order"] = submission_df["cell_order"].fillna("")

out_path = Path("submission.csv")
submission_df.to_csv(out_path, index=False)

print("Wrote:", out_path.resolve())
print(submission_df.head())

assert out_path.exists() and out_path.suffix == ".csv"
assert submission_df.shape[0] == sample_df.shape[0]
assert submission_df["cell_order"].dtype == object
assert submission_df["cell_order"].str.len().ge(0).all()

Wrote: /kaggle/working/submission.csv
               id                                         cell_order
0  00015c83e2717b  924c9f0f c417225b cd61f6d1 51e3cd89 f2c750d3 f...
1  0001bdd4021779  58bf360b 3fdc37be 073782ca 8ea7263c 7fde4f04 8...
2  000757b90aaca0  6243c12c 8f84d7a9 72821d9a eb6ca769 d3db4f3e 3...
3  000a2f5243e1ca  1d968d84 5774aca9 2ddb979d dc18005d 3c7d2db1 2...
4  000c1e0e45bb25  b0b0d7b6 3255fff2 d6c42868 dd65465b 07c5250a 3...
